# Sujet 07 — Santander Transaction · Séance 1

Séance : 1
Groupe : [à compléter]

| Porteur # | Nom | Étapes assignées |
|---|---|---|
| P1 | [à compléter] | 1.1 → 1.8 |
| P2 | Prosper [ajouter le nom de famille] | 1.2 → 1.3 → 1.5 → 1.10 |
| P3 | Prosper [ajouter le nom de famille] | 1.6 → 1.4 (attend 1.3) → 1.9 |
| P4 | [à compléter] | 1.7 → 1.11 → 1.12 |

**Déclaration d'usage de l'IA.** Le code de ce carnet a été écrit avec l'aide de Claude (Anthropic), puis exécuté et relu par l'étudiant. Les chiffres notés sous chaque étape sont ceux affichés par nos exécutions sur nos fichiers.

Ce carnet couvre les étapes de **P2 et P3**. Les cellules sont dans l'ordre logique : l'étape 1.10 (P2) vient après 1.9 (P3), car elle utilise le même découpage.

## Préparation
Imports, paramètres et liste des 200 variables. Pour changer de dossier de données, on modifie `DATA_DIR` ici, nulle part ailleurs.

In [ ]:
import os

import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

DATA_DIR = "."   # dossier qui contient train.csv (sur Kaggle : /kaggle/input/<nom-du-jeu>)
SEED = 42
VARS = [f"var_{i}" for i in range(200)]

## Étape 1.2 — Charger la table (P2)
**Je fais :** je lis les fichiers et j'affiche leurs dimensions.
**Je produis :** le nombre de lignes et de colonnes.

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
a_decider = pd.read_csv(os.path.join(DATA_DIR, "a_decider.csv"))
print("train     :", train.shape[0], "lignes,", train.shape[1], "colonnes")
print("a_decider :", a_decider.shape[0], "lignes,", a_decider.shape[1], "colonnes")
print("Colonnes de train absentes de a_decider :", set(train.columns) - set(a_decider.columns))

**Résultat mesuré :** `train.csv` 160 000 × 202 ; `a_decider.csv` 40 000 × 201 ; seule `target` manque dans `a_decider`.

## Étape 1.3 — Vérifier les types (P2)
**Je fais :** je compare ce que pandas croit et ce que les colonnes contiennent. Une colonne décimale peut cacher des codes (0, -1, -999...) ou des entiers déguisés.
**Je produis :** la liste des colonnes dont j'ai changé le type, avec la raison.

In [ ]:
print(train.dtypes.astype(str).value_counts())

X = train[VARS]
entieres = [c for c in VARS if (X[c] == X[c].round()).all()]
print("\nVariables dont toutes les valeurs sont entières :", len(entieres))

CODES = [0, -1, -9, -99, -999, -9999, 9999, 99999]
print("Variables contenant un code suspect :", int((X.isin(CODES).sum() > 0).sum()))
for c in CODES:
    print("  valeur", c, ":", int((X == c).to_numpy().sum()), "fois")

**Résultat mesuré :** 200 colonnes décimales, `ID_code` texte, `target` entier. Aucune variable entière. Codes trouvés : 0 (111 fois), -1 (78 fois), -9 (28 fois), les autres 0 fois, sur environ 32 millions de valeurs : coïncidences d'arrondi, pas de marque d'absence. **Aucune colonne dont le type a été changé.**

## Étape 1.5 — Chercher les valeurs impossibles (P2)
**Je fais :** les variables sont anonymes, donc je ne juge pas par le métier. Je cherche des valeurs infinies, des valeurs extrêmes et une valeur qui se répète trop (marque d'absence possible).
**Je produis :** la liste des valeurs trouvées et le traitement appliqué.

In [ ]:
print("Valeurs infinies :", int((X.abs() == float("inf")).to_numpy().sum()))
print("Minimum global :", round(float(X.min().min()), 2), "| Maximum global :", round(float(X.max().max()), 2))
print("Variables avec des valeurs négatives :", int((X.min() < 0).sum()))

z = (X - X.mean()) / X.std()
print("Valeurs à plus de 5 écarts-types :", int((z.abs() > 5).to_numpy().sum()))

# valeur la plus répétée de chaque variable
repetees = pd.Series({c: int(X[c].value_counts().iloc[0]) for c in VARS}).sort_values(ascending=False)
print("\nVariables avec la valeur la plus répétée :")
print(repetees.head(5))

# la valeur répétée de var_68 est-elle liée à la cible ?
valeur = X["var_68"].value_counts().index[0]
masque = X["var_68"] == valeur
print("\nvar_68 =", valeur, ":", int(masque.sum()), "lignes")
print("  taux de 1 sur ces lignes :", round(train.loc[masque, "target"].mean() * 100, 2), "%")
print("  taux de 1 sur les autres :", round(train.loc[~masque, "target"].mean() * 100, 2), "%")

**Résultat mesuré :** aucune valeur infinie ; aucune valeur à plus de 5 écarts-types (maximum observé : 4,5). La valeur la plus répétée est 5,0183 dans `var_68` (889 fois, 0,56 %) ; le taux de cible y est de 10,24 % contre 10,05 % ailleurs, donc pas de lien avec la cible. **Traitement : aucun.**
Écart avec l'énoncé : étendue mesurée de −90,25 à +74,03 avec 131 variables négatives, contre −73,8 à +65,4 et 124 annoncés. Nous retenons nos mesures.

## Étape 1.6 — Compter les trous (P3)
**Je fais :** je mesure la proportion de valeurs manquantes, colonne par colonne.
**Je produis :** le tableau trié du plus rempli au moins rempli, et ce que j'observe sur la répartition des trous.

In [ ]:
rempli = (1 - train.isna().mean()).sort_values(ascending=False)
print(rempli.head(3))
print("...")
print(rempli.tail(3))
print("\nColonnes entièrement remplies :", int((rempli == 1).sum()), "sur", len(rempli))
print("Valeurs manquantes dans a_decider :", int(a_decider.isna().sum().sum()))

**Résultat mesuré :** les 202 colonnes sont remplies à 100 %, dans `train` comme dans `a_decider`. Il n'y a donc aucune répartition de trous à observer.

## Étape 1.4 — Compter les valeurs différentes (P3, après 1.3)
**Je fais :** je compte les valeurs distinctes de chaque colonne. Cette étape attend 1.3, car il faut connaître les types.
**Je produis :** le tableau trié, et les colonnes qu'on ne peut pas encoder telles quelles.

In [ ]:
distinctes = train.nunique().sort_values()
print("Les 5 moins variées :")
print(distinctes.head(5))
print("\nLes 3 plus variées :")
print(distinctes.tail(3))
print("\nColonnes constantes :", list(distinctes[distinctes <= 1].index))

**Résultat mesuré :** `ID_code` a 160 000 valeurs distinctes (une par client) et ne s'encode pas tel quel. La variable la moins variée est `var_68` (447 valeurs), puis `var_91` (7 825) ; la plus variée est `var_45` (140 250). Aucune colonne constante.

## Étape 1.9 — Décrire la cible (P3)
**Je fais :** je mets de côté 25 % de `train` pour valider, en gardant la même proportion de 1 grâce à `stratify`, puis je décris la cible dans les deux parties.
**Je produis :** la répartition et les deux taux.

In [ ]:
tr, va = train_test_split(train, test_size=0.25, stratify=train["target"], random_state=SEED)

print(train["target"].value_counts())
print("\nApprentissage :", len(tr), "lignes, taux de 1 =", round(tr["target"].mean() * 100, 2), "%")
print("Validation    :", len(va), "lignes, taux de 1 =", round(va["target"].mean() * 100, 2), "%")

**Résultat mesuré :** 143 922 zéros et 16 078 uns, soit 10,05 %. Apprentissage : 120 000 lignes, 10,05 %. Validation : 40 000 lignes, 10,05 %. Les deux taux sont identiques parce que le découpage est stratifié. `a_decider` n'a pas de cible.

## Étape 1.10 — Mesurer le plancher (P2, après 1.9)
**Je fais :** je construis une règle qui n'utilise **aucune variable** et je mesure son score sur la validation. C'est le score que le modèle devra battre.
**Je produis :** la règle, la valeur qu'elle produit, et son score.

In [ ]:
taux_tr = tr["target"].mean()

# Règle : tout client reçoit le même score, le taux de 1 de l'apprentissage
score_constant = pd.Series(taux_tr, index=va.index)
auc = roc_auc_score(va["target"], score_constant)

# Décision associée : on répond "pas de transaction" pour tout le monde
exactitude = accuracy_score(va["target"], pd.Series(0, index=va.index))

print("Règle : même probabilité pour tous =", round(taux_tr * 100, 2), "%")
print("AUC en validation :", round(auc, 4))
print("Exactitude si on répond toujours 0 :", round(exactitude * 100, 2), "%")

**Résultat mesuré :** AUC = 0,5000 ; exactitude = 89,95 % en répondant toujours « non », mais aucun positif retrouvé. Un score identique pour tous donne toujours un AUC de 0,5, quelle que soit la valeur choisie.

**Pour mémoire.** Le 0,5639 de l'énoncé correspond à une règle qui utilise une variable. Mesure de la meilleure variable seule (`var_81`) sur la validation :

In [ ]:
a = roc_auc_score(va["target"], va["var_81"])
print("AUC de var_81 seule en validation :", round(max(a, 1 - a), 4))

**Résultat mesuré :** environ 0,563, proche du 0,5639 de l'énoncé. C'est une approximation de la règle de la banque, pas une certitude.